# LLM Cycle-Geometry — Task 15 Full-Model Run (Colab)

Runs Part 1 (circle reproduction, the validation gate) and Part 2 (the AUC comparison ladder) against the real `google/gemma-2-2b` model.

**Before running:** Runtime → Change runtime type → select a GPU (e.g. T4).

## 1. Mount Google Drive and install the package wheel

The wheel (`networkgeometry-0.1.0-py3-none-any.whl`) lives in your Drive folder `NetworkGeometry-Colab/`. This mounts your Drive (one browser authorization click) and installs directly from there — no manual upload needed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/NetworkGeometry-Colab'
WHEEL_PATH = f'{DRIVE_DIR}/networkgeometry-0.1.0-py3-none-any.whl'
print('Wheel path:', WHEEL_PATH)

In [ ]:
!pip install -q "{WHEEL_PATH}"
!pip install -q transformer_lens

## 2. Authenticate with Hugging Face

Runs an interactive login widget — paste your token (the same one used locally, or a fresh one). You must have already accepted the license at https://huggingface.co/google/gemma-2-2b under this account.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from huggingface_hub import model_info
info = model_info('google/gemma-2-2b')
print('Access confirmed:', info.id, '| gated:', info.gated)

## 3. Load the model (GPU, standard/processed loading path)

On Colab's GPU + Linux environment, the standard `from_pretrained` path (with TransformerLens's weight processing — LayerNorm folding, `center_writing_weights`, `center_unembed`) is expected to work without the crash seen on the local Windows/CPU environment. This is the scientifically faithful loading path — prefer it over `from_pretrained_no_processing`.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())

from networkgeometry.extraction.activations import load_model
import time
t0 = time.perf_counter()
model = load_model('google/gemma-2-2b', device='cuda' if torch.cuda.is_available() else 'cpu')
print(f'Model loaded in {time.perf_counter()-t0:.1f}s')

**If this cell crashes or errors on Colab too:** fall back to `from_pretrained_no_processing` (see the commented cell below) and note in the findings memo that the no-processing loading path was used, per the documented methodological caveat (it skips `center_writing_weights`, which zeroes a residual-stream direction that subsequent LayerNorms discard anyway — a small, likely-negligible deviation, but worth recording).

In [ ]:
# Fallback only if the cell above fails:
# from transformer_lens import HookedTransformer
# model = HookedTransformer.from_pretrained_no_processing('google/gemma-2-2b', device='cuda' if torch.cuda.is_available() else 'cpu')

## 4. Run Part 1 — circle reproduction (the validation gate)

Per spec §4.5: if month/day circularity doesn't reproduce cleanly here, treat it as a pipeline bug and debug before trusting Part 2 — do not proceed on a weak Part 1 result.

In [ ]:
from networkgeometry.run import run_part1
import time

LAYERS = list(range(26))
OUT_DIR = 'results'

t0 = time.perf_counter()
part1_results = run_part1(model, LAYERS, OUT_DIR)
print(f'Part 1 done in {time.perf_counter()-t0:.1f}s')

for name, layer_results in part1_results.items():
    print(f'\n=== {name} ===')
    for lc in layer_results:
        print(f'  layer {lc.layer:2d}: residual={lc.normalized_residual:.4f}  '
              f'angular_order={lc.angular_order:.4f}  top2_var_ratio={lc.top2_variance_ratio:.4f}')

**Stop and inspect before continuing.** Look for layers where `angular_order` and `top2_variance_ratio` are both high (>0.9-ish) for both day and month — those are the clean, reproduced-circle layers. `results/part1_summary.json` now has these numbers on disk too.

## 5. Plot Part 1 manifolds (optional, for visual inspection)

In [ ]:
from networkgeometry.figures.part1_plots import plot_circularity_by_layer
import matplotlib.pyplot as plt
from IPython.display import Image, display

for name, layer_results in part1_results.items():
    path = plot_circularity_by_layer(layer_results, f'{OUT_DIR}/{name}_circularity_by_layer.png')
    display(Image(filename=path))

## 6. Run Part 2 — the AUC comparison ladder

Set `LAYERS` below to the gate-passing layers you identified in Part 1 (or leave as the same full range — `run_part2` applies its own within-structure gate internally per layer, so passing all 26 is safe, just slower).

In [ ]:
from networkgeometry.run import run_part2

t0 = time.perf_counter()
part2_results = run_part2(model, LAYERS, OUT_DIR)
print(f'Part 2 done in {time.perf_counter()-t0:.1f}s')

for lr in part2_results:
    print(f'\nlayer {lr.layer}: within={lr.within:.4f} gate_passed={lr.gate_passed}')
    for target, stats in lr.crosses.items():
        print(f'  -> {target}: auc={stats["auc"]:.4f} sem={stats["sem"]:.4f} '
              f'raw_p={stats["raw_p"]:.4f} fdr_p={stats["fdr_p"]:.4f} bonferroni_p={stats["bonferroni_p"]:.4f}')

`results/summary.json` now has the full ladder on disk.

## 7. Plot Part 2 results

In [ ]:
from networkgeometry.figures.part2_plots import plot_auc_by_layer

path = plot_auc_by_layer(part2_results, ['month', 'years', 'hierarchy', 'flat'], f'{OUT_DIR}/auc_by_layer.png')
display(Image(filename=path))

## 8. Download results back to your machine

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('task15_results', 'zip', OUT_DIR)
files.download('task15_results.zip')

# Also persist a copy to the same Drive folder
shutil.copy('task15_results.zip', f'{DRIVE_DIR}/task15_results.zip')
print(f'Results also saved to Drive: {DRIVE_DIR}/task15_results.zip')